# Pre-entrega 1: Cliente de LLM robusto y asíncrono

Cliente unificado y asíncrono para OpenAI, Anthropic y Gemini: una sola interfaz, proveedores intercambiables, streaming, validación con Pydantic y errores controlados.

Gemini se suma como tercer proveedor porque su API tiene nivel gratuito, lo que permite probar todo el flujo sin tarjeta.

In [1]:
!pip install -q openai anthropic google-genai pydantic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 17.1 MB/s eta 0:00:00


In [4]:
import os
from getpass import getpass


def leer_secreto(nombre: str) -> str | None:
    """Lee un secreto de Colab. Devuelve None si no existe o no hay acceso."""
    try:
        from google.colab import userdata

        for _ in range(2):  # el primer intento tras dar acceso puede devolver un dato raro
            valor = userdata.get(nombre)
            if isinstance(valor, str):
                return valor
    except Exception:
        pass
    return None


def cargar_clave(nombre: str) -> None:
    """Busca la clave en Secretos de Colab; si no está, la pide por teclado (Enter = omitir)."""
    if os.environ.get(nombre):
        print(f"OK: {nombre} ya estaba cargada")
        return
    valor = leer_secreto(nombre)
    if valor is None:
        valor = getpass(f"{nombre} (Enter para omitir): ")
    valor = valor.strip()
    if valor:
        os.environ[nombre] = valor
        print(f"OK: {nombre} cargada")
    else:
        print(f"Aviso: {nombre} omitida")


for nombre in ("GOOGLE_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
    cargar_clave(nombre)

# Variable de configuración: qué proveedor usa AsyncLLMManager por defecto
os.environ.setdefault("LLM_PROVIDER", "gemini")

OK: GOOGLE_API_KEY cargada
OPENAI_API_KEY (Enter para omitir): ··········
Aviso: OPENAI_API_KEY omitida
ANTHROPIC_API_KEY (Enter para omitir): ··········
Aviso: ANTHROPIC_API_KEY omitida


'gemini'

In [5]:
from enum import StrEnum
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field


class Provider(StrEnum):
    OPENAI = "openai"
    ANTHROPIC = "anthropic"
    GEMINI = "gemini"


class ChatMessage(BaseModel):
    """Un mensaje de la conversación."""

    role: Literal["system", "user", "assistant"]
    content: str = Field(min_length=1)


class GenerationConfig(BaseModel):
    """Parámetros de entrada del modelo. Pydantic rechaza valores fuera de rango."""

    model_config = ConfigDict(extra="forbid")  # un parámetro mal escrito es un error, no se ignora

    # None = usar el modelo por defecto del cliente
    model: str | None = None
    # None = no enviar temperatura (algunos modelos de razonamiento no la aceptan)
    temperature: float | None = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=1024, gt=0, le=8192)


class ErrorType(StrEnum):
    VALIDATION = "validation"
    AUTHENTICATION = "authentication"
    RATE_LIMIT = "rate_limit"
    TIMEOUT = "timeout"
    CONNECTION = "connection"
    API_ERROR = "api_error"
    UNKNOWN = "unknown"


class ErrorInfo(BaseModel):
    """Error controlado: lo que devolvemos en lugar de dejar escapar una excepción."""

    type: ErrorType
    message: str
    provider: Provider
    status_code: int | None = None
    retryable: bool = False


class Usage(BaseModel):
    input_tokens: int
    output_tokens: int


class ModelResponse(BaseModel):
    """Respuesta unificada del modo normal. Si falló, `error` viene completo."""

    provider: Provider
    model: str
    content: str = ""
    finish_reason: str | None = None
    usage: Usage | None = None
    error: ErrorInfo | None = None

    @property
    def ok(self) -> bool:
        return self.error is None


class StreamChunk(BaseModel):
    """Un fragmento del stream. Si algo falla a mitad de camino, llega un chunk con `error`."""

    content: str = ""
    error: ErrorInfo | None = None

    @property
    def ok(self) -> bool:
        return self.error is None

In [6]:
from pydantic import ValidationError

for descripcion, crear in [
    ("temperatura fuera de rango", lambda: GenerationConfig(temperature=5)),
    ("rol inexistente", lambda: ChatMessage(role="robot", content="hola")),
    ("parámetro mal escrito", lambda: GenerationConfig(temperatura=0.5)),
]:
    try:
        crear()
    except ValidationError as exc:
        print(f"Rechazado ({descripcion}): {exc.errors()[0]['msg']}")

Rechazado (temperatura fuera de rango): Input should be less than or equal to 2
Rechazado (rol inexistente): Input should be 'system', 'user' or 'assistant'
Rechazado (parámetro mal escrito): Extra inputs are not permitted


In [7]:
from abc import ABC, abstractmethod
from collections.abc import AsyncIterator, Sequence
from typing import Any, ClassVar

from pydantic import TypeAdapter

_MESSAGES = TypeAdapter(list[ChatMessage])

# (tipo de error, mensaje, ¿tiene sentido reintentar?)
Clasificacion = tuple[ErrorType, str, bool]


class BaseLLMClient(ABC):
    provider: ClassVar[Provider]
    _client: Any

    def __init__(self, default_model: str) -> None:
        self.default_model = default_model

    # ---------- API pública ----------

    async def generate(
        self,
        messages: Sequence[ChatMessage | dict],
        config: GenerationConfig | None = None,
    ) -> ModelResponse:
        """Modo normal: espera la respuesta completa."""
        config = config or GenerationConfig()
        model = config.model or self.default_model
        try:
            validated = self._validate(messages)
            return await self._generate(validated, config, model)
        except Exception as exc:  # CancelledError no es Exception: la cancelación sigue funcionando
            return ModelResponse(provider=self.provider, model=model, error=self._to_error(exc))

    async def stream(
        self,
        messages: Sequence[ChatMessage | dict],
        config: GenerationConfig | None = None,
    ) -> AsyncIterator[StreamChunk]:
        """Modo streaming: generador asíncrono que entrega el texto a medida que llega."""
        config = config or GenerationConfig()
        model = config.model or self.default_model
        try:
            validated = self._validate(messages)
            async for text in self._stream(validated, config, model):
                yield StreamChunk(content=text)
        except Exception as exc:
            yield StreamChunk(error=self._to_error(exc))

    async def aclose(self) -> None:
        """Cierra las conexiones HTTP del SDK (si el SDK lo permite)."""
        cerrar = getattr(self._client, "close", None)
        if cerrar is not None:
            await cerrar()

    # ---------- A implementar por cada proveedor ----------

    @abstractmethod
    async def _generate(
        self, messages: list[ChatMessage], config: GenerationConfig, model: str
    ) -> ModelResponse: ...

    @abstractmethod
    def _stream(
        self, messages: list[ChatMessage], config: GenerationConfig, model: str
    ) -> AsyncIterator[str]: ...

    @abstractmethod
    def _clasificar(self, exc: Exception) -> Clasificacion: ...

    # ---------- Helpers compartidos ----------

    @staticmethod
    def _validate(messages: Sequence[ChatMessage | dict]) -> list[ChatMessage]:
        validated = _MESSAGES.validate_python(list(messages))
        if not validated:
            raise ValueError("La lista de mensajes está vacía.")
        return validated

    def _to_error(self, exc: Exception) -> ErrorInfo:
        """Traduce cualquier excepción a un error estructurado."""
        if isinstance(exc, ValueError):  # incluye el ValidationError de Pydantic
            tipo, mensaje, retryable = ErrorType.VALIDATION, f"Entrada inválida: {exc}", False
        else:
            tipo, mensaje, retryable = self._clasificar(exc)
        codigo = getattr(exc, "status_code", None) or getattr(exc, "code", None)
        return ErrorInfo(
            type=tipo,
            message=mensaje,
            provider=self.provider,
            status_code=codigo if isinstance(codigo, int) else None,
            retryable=retryable,
        )


def clasificar_estilo_openai(sdk: Any, exc: Exception) -> Clasificacion:
    """Los SDK de OpenAI y Anthropic nombran igual sus excepciones: se mapean una sola vez.

    El orden importa: APITimeoutError hereda de APIConnectionError,
    y RateLimitError / AuthenticationError heredan de APIStatusError.
    """
    if isinstance(exc, sdk.APITimeoutError):
        return ErrorType.TIMEOUT, "La API tardó demasiado en responder.", True
    if isinstance(exc, sdk.APIConnectionError):
        return ErrorType.CONNECTION, "No se pudo conectar con la API (error de red).", True
    if isinstance(exc, sdk.RateLimitError):
        return ErrorType.RATE_LIMIT, "Límite de tasa o cuota alcanzado.", True
    if isinstance(exc, (sdk.AuthenticationError, sdk.PermissionDeniedError)):
        return ErrorType.AUTHENTICATION, "API key inválida o sin permisos.", False
    if isinstance(exc, sdk.APIStatusError):
        return (
            ErrorType.API_ERROR,
            f"La API respondió con error {exc.status_code}: {exc.message}",
            exc.status_code >= 500,
        )
    return ErrorType.UNKNOWN, f"Error inesperado: {exc!r}", False

In [8]:
import openai
from openai import AsyncOpenAI


class OpenAIClient(BaseLLMClient):
    provider = Provider.OPENAI

    def __init__(
        self,
        api_key: str,
        default_model: str = "gpt-4.1-mini",
        timeout: float = 30.0,
        max_retries: int = 2,
    ) -> None:
        super().__init__(default_model)
        self._client = AsyncOpenAI(api_key=api_key, timeout=timeout, max_retries=max_retries)

    @staticmethod
    def _params(messages: list[ChatMessage], config: GenerationConfig, model: str) -> dict[str, Any]:
        params: dict[str, Any] = {
            "model": model,
            "messages": [m.model_dump() for m in messages],
            "max_completion_tokens": config.max_tokens,
        }
        if config.temperature is not None:
            params["temperature"] = config.temperature
        return params

    async def _generate(
        self, messages: list[ChatMessage], config: GenerationConfig, model: str
    ) -> ModelResponse:
        resp = await self._client.chat.completions.create(**self._params(messages, config, model))
        choice = resp.choices[0]
        usage = None
        if resp.usage:
            usage = Usage(
                input_tokens=resp.usage.prompt_tokens,
                output_tokens=resp.usage.completion_tokens,
            )
        return ModelResponse(
            provider=self.provider,
            model=resp.model,
            content=choice.message.content or "",
            finish_reason=choice.finish_reason,
            usage=usage,
        )

    async def _stream(
        self, messages: list[ChatMessage], config: GenerationConfig, model: str
    ) -> AsyncIterator[str]:
        stream = await self._client.chat.completions.create(
            **self._params(messages, config, model), stream=True
        )
        async for chunk in stream:
            if chunk.choices and (delta := chunk.choices[0].delta.content):
                yield delta

    def _clasificar(self, exc: Exception) -> Clasificacion:
        return clasificar_estilo_openai(openai, exc)

In [9]:
import anthropic
from anthropic import AsyncAnthropic


class AnthropicClient(BaseLLMClient):
    provider = Provider.ANTHROPIC

    def __init__(
        self,
        api_key: str,
        default_model: str = "claude-haiku-4-5-20251001",
        timeout: float = 30.0,
        max_retries: int = 2,
    ) -> None:
        super().__init__(default_model)
        self._client = AsyncAnthropic(api_key=api_key, timeout=timeout, max_retries=max_retries)

    @staticmethod
    def _params(messages: list[ChatMessage], config: GenerationConfig, model: str) -> dict[str, Any]:
        system = "\n\n".join(m.content for m in messages if m.role == "system")
        chat = [{"role": m.role, "content": m.content} for m in messages if m.role != "system"]

        params: dict[str, Any] = {"model": model, "messages": chat, "max_tokens": config.max_tokens}
        if system:
            params["system"] = system
        if config.temperature is not None:
            params["temperature"] = min(config.temperature, 1.0)
        return params

    async def _generate(
        self, messages: list[ChatMessage], config: GenerationConfig, model: str
    ) -> ModelResponse:
        resp = await self._client.messages.create(**self._params(messages, config, model))
        return ModelResponse(
            provider=self.provider,
            model=resp.model,
            content="".join(block.text for block in resp.content if block.type == "text"),
            finish_reason=resp.stop_reason,
            usage=Usage(
                input_tokens=resp.usage.input_tokens,
                output_tokens=resp.usage.output_tokens,
            ),
        )

    async def _stream(
        self, messages: list[ChatMessage], config: GenerationConfig, model: str
    ) -> AsyncIterator[str]:
        async with self._client.messages.stream(**self._params(messages, config, model)) as stream:
            async for text in stream.text_stream:
                yield text

    def _clasificar(self, exc: Exception) -> Clasificacion:
        return clasificar_estilo_openai(anthropic, exc)

In [10]:
import asyncio

import httpx
from google import genai
from google.genai import errors as genai_errors
from google.genai import types


class GeminiClient(BaseLLMClient):
    provider = Provider.GEMINI

    def __init__(
        self,
        api_key: str,
        default_model: str = "gemini-flash-latest",
        timeout: float = 30.0,
        max_retries: int = 2,
    ) -> None:
        super().__init__(default_model)
        self._max_retries = max_retries
        self._client = genai.Client(
            api_key=api_key,
            http_options=types.HttpOptions(timeout=int(timeout * 1000)),  # en milisegundos
        )

    @staticmethod
    def _params(messages: list[ChatMessage], config: GenerationConfig, model: str) -> dict[str, Any]:
        system = "\n\n".join(m.content for m in messages if m.role == "system")
        contents = [
            types.Content(
                role="model" if m.role == "assistant" else "user",
                parts=[types.Part(text=m.content)],
            )
            for m in messages
            if m.role != "system"
        ]
        return {
            "model": model,
            "contents": contents,
            "config": types.GenerateContentConfig(
                temperature=config.temperature,
                max_output_tokens=config.max_tokens,
                system_instruction=system or None,
            ),
        }

    async def _generate(
        self, messages: list[ChatMessage], config: GenerationConfig, model: str
    ) -> ModelResponse:
        params = self._params(messages, config, model)
        for intento in range(self._max_retries + 1):
            try:
                resp = await self._client.aio.models.generate_content(**params)
                break
            except Exception as exc:
                reintentable = self._clasificar(exc)[2]
                if intento == self._max_retries or not reintentable:
                    raise
                await asyncio.sleep(2**intento)  # backoff: 1s, 2s, 4s...

        usage = None
        meta = resp.usage_metadata
        if meta and meta.prompt_token_count is not None:
            usage = Usage(
                input_tokens=meta.prompt_token_count,
                output_tokens=meta.candidates_token_count or 0,
            )
        finish = resp.candidates[0].finish_reason if resp.candidates else None
        return ModelResponse(
            provider=self.provider,
            model=model,
            content=resp.text or "",
            finish_reason=getattr(finish, "name", None),
            usage=usage,
        )

    async def _stream(
        self, messages: list[ChatMessage], config: GenerationConfig, model: str
    ) -> AsyncIterator[str]:
        stream = await self._client.aio.models.generate_content_stream(
            **self._params(messages, config, model)
        )
        async for chunk in stream:
            if chunk.text:
                yield chunk.text

    def _clasificar(self, exc: Exception) -> Clasificacion:
        if isinstance(exc, genai_errors.APIError):
            codigo = exc.code if isinstance(exc.code, int) else 0
            if codigo == 429:
                return ErrorType.RATE_LIMIT, "Límite de tasa o cuota alcanzado.", True
            if codigo in (401, 403) or "API key" in str(exc):
                return ErrorType.AUTHENTICATION, "API key inválida o sin permisos.", False
            detalle = getattr(exc, "message", None) or str(exc)
            return ErrorType.API_ERROR, f"La API respondió con error {codigo}: {detalle}", codigo >= 500
        if isinstance(exc, (httpx.TimeoutException, TimeoutError)):
            return ErrorType.TIMEOUT, "La API tardó demasiado en responder.", True
        if isinstance(exc, (httpx.TransportError, OSError)):
            return ErrorType.CONNECTION, "No se pudo conectar con la API (error de red).", True
        return ErrorType.UNKNOWN, f"Error inesperado: {exc!r}", False

    async def aclose(self) -> None:
        cerrar = getattr(self._client.aio, "aclose", None)
        if cerrar is not None:
            await cerrar()

In [11]:
from typing import Self


class ConfigurationError(Exception):
    """Configuración inválida al arrancar (proveedor desconocido o falta la API key)."""


# proveedor -> (clase, variable de la API key, variable del modelo)
_REGISTRY: dict[Provider, tuple[type[BaseLLMClient], str, str]] = {
    Provider.OPENAI: (OpenAIClient, "OPENAI_API_KEY", "OPENAI_MODEL"),
    Provider.ANTHROPIC: (AnthropicClient, "ANTHROPIC_API_KEY", "ANTHROPIC_MODEL"),
    Provider.GEMINI: (GeminiClient, "GOOGLE_API_KEY", "GEMINI_MODEL"),
}


class AsyncLLMManager:
    def __init__(self, provider: Provider | str | None = None, *, api_key: str | None = None) -> None:
        name = (provider or os.getenv("LLM_PROVIDER") or "gemini").strip().lower()
        try:
            self.provider = Provider(name)
        except ValueError:
            valid = ", ".join(p.value for p in Provider)
            raise ConfigurationError(f"Proveedor desconocido: '{name}'. Opciones: {valid}.") from None

        client_cls, key_var, model_var = _REGISTRY[self.provider]

        api_key = api_key or os.getenv(key_var)
        if not api_key:
            raise ConfigurationError(f"Falta la variable de entorno {key_var}.")

        try:
            timeout = float(os.getenv("LLM_TIMEOUT", "30"))
            max_retries = int(os.getenv("LLM_MAX_RETRIES", "2"))
        except ValueError:
            raise ConfigurationError("LLM_TIMEOUT y LLM_MAX_RETRIES deben ser numéricos.") from None

        kwargs: dict = {"api_key": api_key, "timeout": timeout, "max_retries": max_retries}
        if model := os.getenv(model_var):
            kwargs["default_model"] = model

        self._client: BaseLLMClient = client_cls(**kwargs)

    @property
    def model(self) -> str:
        return self._client.default_model

    async def generate(
        self,
        messages: Sequence[ChatMessage | dict],
        config: GenerationConfig | None = None,
    ) -> ModelResponse:
        return await self._client.generate(messages, config)

    async def stream(
        self,
        messages: Sequence[ChatMessage | dict],
        config: GenerationConfig | None = None,
    ) -> AsyncIterator[StreamChunk]:
        async for chunk in self._client.stream(messages, config):
            yield chunk

    async def aclose(self) -> None:
        await self._client.aclose()

    async def __aenter__(self) -> Self:
        return self

    async def __aexit__(self, *exc_info: object) -> None:
        await self.aclose()

In [12]:
PREGUNTA = "¿Qué es la entropía?"
MENSAJES = [
    ChatMessage(role="system", content="Respondé en español, en no más de tres oraciones."),
    ChatMessage(role="user", content=PREGUNTA),
]
CONFIG = GenerationConfig(temperature=0.3, max_tokens=1000)


async def probar(provider: str | None = None) -> None:
    """Hace la misma pregunta en modo normal y en streaming."""
    try:
        llm = AsyncLLMManager(provider)
    except ConfigurationError as exc:
        print(f"[config] {exc}")
        return

    async with llm:
        print(f"===== {llm.provider.value} ({llm.model}) =====")

        print("\n--- Modo normal ---")
        resp = await llm.generate(MENSAJES, CONFIG)
        if resp.ok:
            print(resp.content)
            if resp.usage:
                print(f"[tokens: {resp.usage.input_tokens} entrada / {resp.usage.output_tokens} salida]")
        else:
            print(f"[error controlado] {resp.error.type.value}: {resp.error.message}")

        print("\n--- Modo streaming ---")
        async for chunk in llm.stream(MENSAJES, CONFIG):
            if chunk.ok:
                print(chunk.content, end="", flush=True)
            else:
                print(f"\n[error controlado] {chunk.error.type.value}: {chunk.error.message}")
        print()

In [15]:
os.environ["GEMINI_MODEL"] = "gemini-flash-lite-latest"

In [16]:
await probar()

===== gemini (gemini-flash-lite-latest) =====

--- Modo normal ---
La entropía es una magnitud física que mide el grado de desorden o aleatoriedad molecular en un sistema termodinámico. Según la segunda ley de la termodinámica, la entropía del universo siempre tiende a aumentar con el tiempo en los procesos espontáneos. En términos sencillos, representa la cantidad de energía que ya no está disponible para realizar un trabajo útil.
[tokens: 22 entrada / 76 salida]

--- Modo streaming ---
La entropía es una magnitud física que mide el grado de desorden o aleatoriedad molecular en un sistema termodinámico. Según la segunda ley de la termodinámica, este valor tiende a aumentar con el tiempo en los procesos espontáneos del universo. En términos sencillos, representa la irreversibilidad de los estados físicos y la pérdida de energía útil para realizar trabajo.


In [ ]:
await probar("openai")

In [ ]:
await probar("anthropic")

In [17]:
import time

PREGUNTAS = [
    "¿Qué es un token en un LLM?",
    "¿Qué es la temperatura en un LLM?",
    "¿Qué es el streaming de tokens?",
]


async def demo_concurrencia(provider: str | None = None) -> None:
    try:
        llm = AsyncLLMManager(provider)
    except ConfigurationError as exc:
        print(f"[config] {exc}")
        return

    async with llm:
        inicio = time.perf_counter()
        respuestas = await asyncio.gather(
            *(
                llm.generate([ChatMessage(role="user", content=f"{p} Respondé en una oración.")], CONFIG)
                for p in PREGUNTAS
            )
        )
        total = time.perf_counter() - inicio

    for pregunta, resp in zip(PREGUNTAS, respuestas):
        texto = resp.content.strip() if resp.ok else f"[error controlado] {resp.error.type.value}: {resp.error.message}"
        print(f"- {pregunta}\n  {texto}\n")
    print(f"{len(PREGUNTAS)} consultas concurrentes en {total:.1f} s")


await demo_concurrencia()

- ¿Qué es un token en un LLM?
  En un LLM, un token es la unidad básica de texto (que puede ser una palabra, parte de ella o un signo de puntuación) en la que se divide la información para que el modelo pueda procesarla y comprenderla.

- ¿Qué es la temperatura en un LLM?
  La temperatura en un LLM es un parámetro que controla la aleatoriedad y creatividad en las respuestas del modelo: valores bajos generan respuestas más predecibles y precisas, mientras que valores altos producen resultados más variados y creativos.

- ¿Qué es el streaming de tokens?
  El streaming de tokens es una técnica mediante la cual una inteligencia artificial genera y muestra las palabras o fragmentos de texto en tiempo real, a medida que los va procesando, en lugar de esperar a tener la respuesta completa.

3 consultas concurrentes en 1.3 s


In [18]:
llm_roto = AsyncLLMManager("openai", api_key="sk-clave-invalida-a-proposito")

resp = await llm_roto.generate(MENSAJES, CONFIG)
print("¿El programa siguió vivo? Sí")
print(f"Error capturado (sin crash): {resp.error.type.value} (HTTP {resp.error.status_code}): {resp.error.message}")

print("\nLo mismo en streaming:")
async for chunk in llm_roto.stream(MENSAJES, CONFIG):
    print(chunk.content if chunk.ok else f"[error controlado] {chunk.error.type.value}: {chunk.error.message}")

await llm_roto.aclose()

¿El programa siguió vivo? Sí
Error capturado (sin crash): authentication (HTTP 401): API key inválida o sin permisos.

Lo mismo en streaming:
[error controlado] authentication: API key inválida o sin permisos.
